# NGC 4395 and POX 52

The same workflow as the Fairall 9 tutorial, applied in a loop to two dwarf galaxies with low-mass AGN, NGC 4395 and POX 52. Each target gets its own stellar-mass prior; the redshifts are fixed to their spectroscopic values.

You need internet access and the DSPS SSP file `tempdata.h5` in the repository root.

In [ ]:
import numpyro.distributions as dist
from astropy.table import Table

from jaxsedfit import (
    AGNConfig,
    FilterSet,
    FitConfig,
    GalaxyConfig,
    InferenceConfig,
    JAXSEDFit,
    LikelihoodConfig,
    Observation,
    PhotometryData,
    PriorConfig,
    load_filter_curves,
)
from jaxsedfit.vizier import query_vizier_sed

## 1. Targets and photometry

In [ ]:
targets = {
    "NGC 4395": {"redshift": 0.001064, "log_mass_prior": 8.5},
    "POX 52": {"redshift": 0.0218, "log_mass_prior": 9.0},
}

photometry = {}
for name in targets:
    rows, _, _ = query_vizier_sed(name, radius=5.0)
    photometry[name] = Table(rows=rows)["grahsp_filter", "flux_mjy", "err_mjy", "psf_fwhm_arcsec"]
    print(f"{name}: {len(rows)} bands: {list(photometry[name]['grahsp_filter'])}")

## 2. Fit each target

As for Fairall 9, `use_host_capture_model=True` lets the captured host light vary with each survey's spatial resolution.

In [ ]:
def make_config(name, target, phot):
    return FitConfig(
        observation=Observation(object_id=name, redshift=target["redshift"]),
        photometry=PhotometryData(
            filter_names=list(phot["grahsp_filter"]),
            fluxes=list(phot["flux_mjy"]),
            errors=list(phot["err_mjy"]),
            psf_fwhm_arcsec=list(phot["psf_fwhm_arcsec"]),
        ),
        filters=FilterSet(curves=load_filter_curves(list(phot["grahsp_filter"]))),
        galaxy=GalaxyConfig(dsps_ssp_fn="../tempdata.h5"),
        agn=AGNConfig(agn_type=1),
        likelihood=LikelihoodConfig(use_host_capture_model=True),
        inference=InferenceConfig(method="optax+nuts", map_steps=400, learning_rate=5e-3, num_warmup=100, num_samples=50),
        prior_config=PriorConfig(
            stellar_mass=dist.Normal(target["log_mass_prior"], 1.0),
            host={"ebv_gal": dist.HalfNormal(0.15)},
            agn={"ebv_agn": dist.HalfNormal(0.15)},
        ),
    )


fitters = {}
for name, target in targets.items():
    fitters[name] = JAXSEDFit(make_config(name, target, photometry[name]))
    fitters[name].fit(progress_bar=False)

## 3. Results

In [ ]:
for name, fitter in fitters.items():
    fitter.plot_sed(show=True, title=name)
    print(f"{name}: log M* = {fitter.summary()['log_stellar_mass_median']:.2f}")